In [3]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Cuarentena NYC Taxis") \
    .getOrCreate()

In [4]:
from pyspark.sql.functions import col, lit, coalesce, to_timestamp
 
# Configuración de estabilidad máxima
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
spark.conf.set("spark.sql.parquet.mergeSchema", "false")
spark.conf.set("spark.sql.caseSensitive", "false")
 
print("Iniciando CUARENTENA: Estrategia de Aislamiento Mensual...")
 
RUTA_BASE = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/"
 
def procesar_segmento_blindado(año, mes, tipo):
    # Formateamos el mes a dos dígitos (01, 02...)
    mes_str = str(mes).zfill(2)
    ruta = f"{RUTA_BASE}year={año}/month={mes_str}/type={tipo}/*.parquet"
   
    try:
        # Leemos el mes específico de forma aislada
        df_mes = spark.read.parquet(ruta)
        cols = df_mes.columns
       
        def buscar(lista):
            for c in lista:
                if c in cols: return col(c)
            return lit(None)
 
        # Aplicamos CAST inmediatamente para estandarizar tipos
        return df_mes.select(
            lit(tipo.capitalize()).alias("taxi_color"),
            to_timestamp(coalesce(
                buscar(["tpep_pickup_datetime", "lpep_pickup_datetime", "pickup_datetime", "Trip_Pickup_DateTime"])
            )).alias("pickup_datetime"),
            to_timestamp(coalesce(
                buscar(["tpep_dropoff_datetime", "lpep_dropoff_datetime", "dropoff_datetime", "Trip_Dropoff_DateTime"])
            )).alias("dropoff_datetime"),
            buscar(["trip_distance", "Trip_Distance"]).cast("double").alias("trip_distance"),
            coalesce(buscar(["total_amount", "Total_Amt"]), lit(0.0)).cast("double").alias("total_amount"),
            buscar(["PULocationID"]).cast("long").alias("PULocationID"),
            buscar(["DOLocationID"]).cast("long").alias("DOLocationID")
        )
    except:
        # Si no existe la carpeta para ese mes/tipo, saltamos silenciosamente
        return None
 
try:
    df_acumulado = None
    años = range(2009, 2026)
    meses = range(1, 13)
    tipos = ["yellow", "green"]
 
    print("Procesando malla de datos (mes por mes)...")
   
    for a in años:
        for m in meses:
            for t in tipos:
                df_temp = procesar_segmento_blindado(a, m, t)
               
                if df_temp:
                    if df_acumulado is None:
                        df_acumulado = df_temp
                    else:
                        df_acumulado = df_acumulado.unionByName(df_temp, allowMissingColumns=True)
 
    if df_acumulado is None:
        raise Exception("No se encontraron datos. Verifica la RUTA_BASE.")
 
    # Definición de Cuarentena
    condicion_ok = (
        col("pickup_datetime").isNotNull() &
        col("dropoff_datetime").isNotNull() &
        col("trip_distance").isNotNull() &
        col("total_amount").isNotNull() &
        col("PULocationID").isNotNull() &
        col("DOLocationID").isNotNull()
    )
 
    df_validos = df_acumulado.filter(condicion_ok)
    df_rechazados = df_acumulado.filter(~condicion_ok)
 
    # Guardado final
    ruta_ok = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/datos_con_columnas/"
    ruta_err = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/cuarentena/sin_columnas_kpis/"
 
    print("Escribiendo resultados en el Data Lake...")
    df_validos.write.mode("overwrite").parquet(ruta_ok)
    df_rechazados.write.mode("overwrite").parquet(ruta_err)
 
    print("Proceso finalizado con éxito!")
 
except Exception as e:
    print(f"Error crítico: {e}")

In [5]:
spark.stop()